In [ ]:
!pip install pyspark

import pandas as pd
import numpy as np
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

In [ ]:
try:
    sc = SparkContext.getOrCreate()
    spark = SparkSession.builder.getOrCreate()
    print("Using existing Spark session")
except:
    conf = SparkConf().setAppName("CosmeticsAnalysis").setMaster("local[*]")
    sc = SparkContext(conf=conf)
    spark = SparkSession.builder.appName("CosmeticsAnalysis").getOrCreate()
    print("New Spark session created successfully!")

In [ ]:
# Data Loading and Preprocessing Functions
def load_and_clean_data():
    """Load and clean the cosmetics data using RDD operations"""
    try:

        rdd = sc.textFile("Cosmetics Inc. - Data for Cleaning.csv")


        header = rdd.first()
        data_rdd = rdd.filter(lambda line: line != header).filter(lambda line: line.strip() != "")


        def parse_line(line):
            parts = line.split(',')
            if len(parts) >= 6:
                return parts
            return []

        parsed_rdd = data_rdd.map(parse_line).filter(lambda x: len(x) > 0)

        # Clean and transform data
        def clean_record(record):
            try:

                product_full = record[0]
                price_str = record[1].replace('$', '').replace(',', '')
                client = record[2]
                client_code = record[3]

                #  orders
                orders_str = str(record[4]).replace('"', '').replace("'", "").strip()

                #  total
                total_str = str(record[5]).replace('$', '').replace(',', '').replace('#VALUE!', '0').strip()


                if len(product_full) > 5:

                    product_code = ''
                    category = ''
                    for i, char in enumerate(product_full):
                        if char.isdigit():
                            product_code += char
                        else:
                            category = product_full[i:]
                            break
                else:
                    product_code = product_full
                    category = "Unknown"


                price = float(price_str) if price_str.replace('.', '').replace('-', '').isdigit() else 0.0
                orders = int(orders_str) if orders_str.isdigit() else 0
                total = float(total_str) if total_str.replace('.', '').replace('-', '').isdigit() else 0.0

                return (product_code, category, price, client, client_code, orders, total)
            except Exception as e:
                return None

        cleaned_rdd = parsed_rdd.map(clean_record).filter(lambda x: x is not None)

        return cleaned_rdd
    except Exception as e:
        print(f"Error loading data: {e}")
        return None


In [ ]:
def basic_statistics(rdd):
    """Calculate basic statistics"""
    print("=== BASIC STATISTICS ===")

    # Total records
    total_records = rdd.count()
    print(f"Total Records: {total_records}")

    # Average price
    prices = rdd.map(lambda x: x[2])
    avg_price = prices.mean()
    print(f"Average Price: ${avg_price:.2f}")

    # Total revenue
    total_revenue = rdd.map(lambda x: x[6]).sum()
    print(f"Total Revenue: ${total_revenue:,.2f}")

    # Total orders
    total_orders = rdd.map(lambda x: x[5]).sum()
    print(f"Total Orders: {total_orders}")

    # Number of unique clients
    unique_clients = rdd.map(lambda x: x[3]).distinct().count()
    print(f"Unique Clients: {unique_clients}")

    print("\n")

def client_analysis(rdd):
    """Analyze client data"""
    print("=== CLIENT ANALYSIS ===")

    # Revenue by client
    client_revenue = rdd.map(lambda x: (x[3], x[6])).reduceByKey(lambda a, b: a + b)
    print("Revenue by Client:")
    for client, revenue in client_revenue.sortBy(lambda x: -x[1]).collect():
        print(f"  {client}: ${revenue:,.2f}")

    # Orders by client
    client_orders = rdd.map(lambda x: (x[3], x[5])).reduceByKey(lambda a, b: a + b)
    print("\nOrders by Client:")
    for client, orders in client_orders.sortBy(lambda x: -x[1]).collect():
        print(f"  {client}: {orders} orders")

    print("\n")

In [ ]:
def product_analysis(rdd):
    """Analyze product data"""
    print("=== PRODUCT ANALYSIS ===")

    # Revenue by product category
    category_revenue = rdd.map(lambda x: (x[1], x[6])).reduceByKey(lambda a, b: a + b)
    print("Revenue by Product Category:")
    for category, revenue in category_revenue.sortBy(lambda x: -x[1]).collect():
        print(f"  {category}: ${revenue:,.2f}")

    # Average price by category
    category_price = rdd.map(lambda x: (x[1], (x[2], 1)))\
                       .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))\
                       .mapValues(lambda x: x[0] / x[1])
    print("\nAverage Price by Category:")
    for category, avg_price in category_price.sortBy(lambda x: -x[1]).collect():
        print(f"  {category}: ${avg_price:.2f}")

    print("\n")

In [ ]:
def price_range_analysis(rdd):
    """Analyze products by price ranges"""
    print("=== PRICE RANGE ANALYSIS ===")

    def price_category(price):
        if price < 5:
            return "Budget ($0-5)"
        elif price < 10:
            return "Mid-range ($5-10)"
        elif price < 15:
            return "Premium ($10-15)"
        else:
            return "Luxury ($15+)"

    price_categories = rdd.map(lambda x: (price_category(x[2]), 1))\
                         .reduceByKey(lambda a, b: a + b)

    print("Products by Price Category:")
    for category, count in price_categories.sortBy(lambda x: -x[1]).collect():
        print(f"  {category}: {count} products")

In [ ]:
def top_performers(rdd):
    """Find top performing products and clients"""
    print("=== TOP PERFORMERS ===")

    # Top 5 products by revenue
    top_products = rdd.map(lambda x: ((x[0], x[1]), x[6]))\
                     .reduceByKey(lambda a, b: a + b)\
                     .sortBy(lambda x: -x[1])\
                     .take(5)

    print("Top 5 Products by Revenue:")
    for i, ((code, category), revenue) in enumerate(top_products, 1):
        print(f"  {i}. {code}{category}: ${revenue:,.2f}")

    # Top 3 clients by revenue
    top_clients = rdd.map(lambda x: (x[3], x[6]))\
                    .reduceByKey(lambda a, b: a + b)\
                    .sortBy(lambda x: -x[1])\
                    .take(3)

    print("\nTop 3 Clients by Revenue:")
    for i, (client, revenue) in enumerate(top_clients, 1):
        print(f"  {i}. {client}: ${revenue:,.2f}")

In [ ]:
# Visualization Functions
def create_visualizations(rdd):
    """Create visualizations for the data"""
    try:

        data_list = rdd.collect()
        columns = ['Product_Code', 'Category', 'Price', 'Client', 'Client_Code', 'Orders', 'Total']
        df = pd.DataFrame(data_list, columns=columns)

        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

        # Plot 1: Revenue by Client
        client_revenue = df.groupby('Client')['Total'].sum().sort_values(ascending=False)
        client_revenue.plot(kind='bar', ax=ax1, color='skyblue')
        ax1.set_title('Revenue by Client')
        ax1.set_ylabel('Revenue ($)')
        ax1.tick_params(axis='x', rotation=45)

        # Plot 2: Revenue by Product Category
        category_revenue = df.groupby('Category')['Total'].sum().sort_values(ascending=False)
        category_revenue.plot(kind='bar', ax=ax2, color='lightcoral')
        ax2.set_title('Revenue by Product Category')
        ax2.set_ylabel('Revenue ($)')
        ax2.tick_params(axis='x', rotation=45)

        # Plot 3: Orders Distribution
        df['Orders'].hist(ax=ax3, bins=10, color='lightgreen')
        ax3.set_title('Distribution of Orders')
        ax3.set_xlabel('Number of Orders')
        ax3.set_ylabel('Frequency')

        # Plot 4: Price Distribution
        df['Price'].hist(ax=ax4, bins=10, color='gold')
        ax4.set_title('Distribution of Prices')
        ax4.set_xlabel('Price ($)')
        ax4.set_ylabel('Frequency')

        plt.tight_layout()
        plt.show()

        # Additional scatter plot
        plt.figure(figsize=(10, 6))
        plt.scatter(df['Price'], df['Orders'], alpha=0.6, color='purple', s=50)
        plt.xlabel('Price ($)')
        plt.ylabel('Number of Orders')
        plt.title('Price vs Orders Relationship')
        plt.grid(True, alpha=0.3)


        z = np.polyfit(df['Price'], df['Orders'], 1)
        p = np.poly1d(z)
        plt.plot(df['Price'], p(df['Price']), "r--", alpha=0.8)

        plt.show()

    except Exception as e:
        print(f"Error creating visualizations: {e}")

In [ ]:
# Advanced RDD Operations
def advanced_operations(rdd):
    """Showcase advanced RDD operations"""
    print("\n=== ADVANCED RDD OPERATIONS ===")


    high_value = rdd.filter(lambda x: x[6] > 5000)
    print(f"High-value products (revenue > $5,000): {high_value.count()}")


    revenue_per_order = rdd.map(lambda x: (x[0] + x[1], x[6] / x[5] if x[5] > 0 else 0))
    print("\nRevenue per order for top 5 products:")
    for product, rpo in revenue_per_order.sortBy(lambda x: -x[1]).take(5):
        print(f"  {product}: ${rpo:.2f}")

    # Aggregate:
    category_count = rdd.map(lambda x: (x[1], 1)).reduceByKey(lambda a, b: a + b)
    most_popular = category_count.sortBy(lambda x: -x[1]).first()
    print(f"\nMost popular category: {most_popular[0]} ({most_popular[1]} products)")


    client_avg_order = rdd.map(lambda x: (x[3], (x[6], x[5])))\
                         .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))\
                         .mapValues(lambda x: x[0] / x[1] if x[1] > 0 else 0)

    max_avg_client = client_avg_order.sortBy(lambda x: -x[1]).first()
    print(f"Client with highest average order value: {max_avg_client[0]} (${max_avg_client[1]:.2f})")


In [ ]:
# UI Components
def create_dashboard():
    """Create interactive dashboard"""

    load_button = widgets.Button(description="📁 Load & Clean Data", button_style='primary', layout=widgets.Layout(width='180px'))
    stats_button = widgets.Button(description="📊 Basic Statistics", button_style='info', layout=widgets.Layout(width='150px'))
    analysis_button = widgets.Button(description="🔍 Full Analysis", button_style='success', layout=widgets.Layout(width='150px'))
    viz_button = widgets.Button(description="📈 Visualizations", button_style='warning', layout=widgets.Layout(width='150px'))
    advanced_button = widgets.Button(description="⚡ Advanced Ops", button_style='info', layout=widgets.Layout(width='150px'))
    clear_button = widgets.Button(description="🗑️ Clear Output", button_style='danger', layout=widgets.Layout(width='150px'))


    output = widgets.Output()


    global cleaned_rdd
    cleaned_rdd = None


    def on_load_click(b):
        with output:
            clear_output()
            print("🔄 Loading and cleaning data...")
            global cleaned_rdd
            cleaned_rdd = load_and_clean_data()
            if cleaned_rdd is not None:
                print("✅ Data loaded successfully!")
                print(f"📊 Total records: {cleaned_rdd.count()}")
                print("\n🔍 Sample records:")
                for record in cleaned_rdd.take(5):
                    print(f"   {record}")
            else:
                print("❌ Failed to load data. Please check if the file exists.")

    def on_stats_click(b):
        with output:
            clear_output()
            if cleaned_rdd is not None:
                basic_statistics(cleaned_rdd)
            else:
                print("❌ Please load data first!")

    def on_analysis_click(b):
        with output:
            clear_output()
            if cleaned_rdd is not None:
                client_analysis(cleaned_rdd)
                product_analysis(cleaned_rdd)
                price_range_analysis(cleaned_rdd)
                top_performers(cleaned_rdd)
            else:
                print("❌ Please load data first!")

    def on_viz_click(b):
        with output:
            clear_output()
            if cleaned_rdd is not None:
                create_visualizations(cleaned_rdd)
            else:
                print("❌ Please load data first!")

    def on_advanced_click(b):
        with output:
            clear_output()
            if cleaned_rdd is not None:
                advanced_operations(cleaned_rdd)
            else:
                print("❌ Please load data first!")

    def on_clear_click(b):
        with output:
            clear_output()


    load_button.on_click(on_load_click)
    stats_button.on_click(on_stats_click)
    analysis_button.on_click(on_analysis_click)
    viz_button.on_click(on_viz_click)
    advanced_button.on_click(on_advanced_click)
    clear_button.on_click(on_clear_click)


    buttons_row1 = widgets.HBox([load_button, stats_button, analysis_button])
    buttons_row2 = widgets.HBox([viz_button, advanced_button, clear_button])
    buttons = widgets.VBox([buttons_row1, buttons_row2])


    display(buttons)
    display(output)


print("=== 🎨 COSMETICS DATA ANALYSIS DASHBOARD ===")
print("📋 Instructions:")
print("1. 📁 Upload your CSV file to Colab as 'Cosmetics Inc. - Data for Cleaning.csv'")
print("2. 🖱️ Click 'Load & Clean Data' to process your data")
print("3. 🔍 Use other buttons to perform various analyses")
print("4. 📈 View insights and visualizations")
print("\n")

create_dashboard()


print(f"\n🔧 Spark Context: {sc}")
print(f"🔧 Spark Version: {sc.version}")
print(f"🔧 Master: {sc.master}")